# MapLibreum: Advanced Layers

This notebook demonstrates advanced capabilities like 3D terrain, PMTiles (Cloud-Optimized Vector Tiles), Deck.GL visualizations, and Three.js 3D models.

## 1. 3D Terrain & Globe View

MapLibreum can render real 3D terrain and even display the map as a globe. You can also add interactive `TerrainControl`.

In [ ]:
from maplibreum import Map
from maplibreum.controls import TerrainControl

# Map centered in the Alps with pitch to see 3D effect
m1 = Map(
    center=[6.8652, 45.8326], # Mont Blanc
    zoom=11,
    pitch=60,
    bearing=45,
    # Render the Earth as a globe
    projection="globe"
)

# Use Mapzen/AWS terrain tiles for the 3D elevation data
terrain_source = {
    "type": "raster-dem",
    "url": "https://s3.amazonaws.com/elevation-tiles-prod/terrarium/{z}/{x}/{y}.png",
    "tileSize": 256
}
m1.add_source("terrainSource", terrain_source)

# Configure the map to use this source for its 3D terrain
m1.set_terrain("terrainSource", exaggeration=1.5)

# Allow user to toggle 3D on/off
m1.add_control(TerrainControl(source="terrainSource", exaggeration=1.5))

m1


## 2. PMTiles (Cloud-Optimized Vector Tiles)

PMTiles is a single-file archive format for pyramids of map tiles. MapLibreum natively supports loading these via `PMTilesProtocol` and `PMTilesSource`.

In [ ]:
from maplibreum import Map
from maplibreum.pmtiles import PMTilesProtocol, PMTilesSource

m2 = Map(center=[-74.0060, 40.7128], zoom=12) # NYC

# Register the protocol to handle `pmtiles://` URLs
m2.add_protocol(PMTilesProtocol())

# Load a PMTiles archive from a remote URL
pmtiles_url = "https://r2-public.protomaps.com/protomaps-sample-datasets/cb_2018_us_zcta510_500k.pmtiles"
source = PMTilesSource(pmtiles_url)

# Note: add_source is typically standard dictionary, but PMTilesSource acts as a helper
m2.add_source("my-pmtiles", source)

m2.add_layer({
    "id": "zip-codes",
    "type": "fill",
    "source": "my-pmtiles",
    "source-layer": "zcta", # depends on the pmtiles internals
    "paint": {
        "fill-color": "#3388ff",
        "fill-opacity": 0.5
    }
})

m2

## 3. Deck.GL Visualizations

MapLibreum can integrate with Deck.GL to render massive datasets and advanced visualizations like hexbins, scatterplots, or arcs over your map.

In [ ]:
from maplibreum import Map
from maplibreum.deckgl import DeckGLLayer

m3 = Map(center=[-122.4194, 37.7749], zoom=11, pitch=45) # SF

# A simple dataset of starting and ending coordinates for trips
flight_data = [
    {"start": [-122.4, 37.7], "end": [-122.5, 37.8], "color": [255, 0, 0]},
    {"start": [-122.3, 37.8], "end": [-122.4, 37.9], "color": [0, 255, 0]}
]

# Create a Deck.GL ArcLayer to visualize paths
arc_layer = DeckGLLayer(
    id="my-arc-layer",
    layer_type="ArcLayer",
    data=flight_data,
    getSourcePosition="@@=d.start",
    getTargetPosition="@@=d.end",
    getSourceColor="@@=d.color",
    getTargetColor="@@=d.color",
    getWidth=5
)

arc_layer.add_to(m3)

m3

## 4. Three.js 3D Models

With `ThreeJSLayer`, you can place glTF/glb 3D models at specific real-world coordinates on the map. It can even optionally handle vertical placement on 3D terrain.

In [ ]:
from maplibreum import Map
from maplibreum.threejs import ThreeJSLayer

m4 = Map(center=[2.2945, 48.8584], zoom=16, pitch=60) # Eiffel Tower area

# Let's add a sample 3D model (e.g., a simple box or drone)
# We'll use a publicly available sample GLB
model_url = 'https://raw.githubusercontent.com/KhronosGroup/glTF-Sample-Models/master/2.0/Box/glTF-Binary/Box.glb'

# Define model location, altitude
model_origin = [2.2945, 48.8584] # Longitude, Latitude
model_altitude = 50 # meters

three_layer = ThreeJSLayer(
    model_uri=model_url,
    model_origin=model_origin,
    model_altitude=model_altitude,
    model_scale=[20, 20, 20], # Scale it up to make it visible
    id="my-3d-model"
)

three_layer.add_to(m4)

m4